# AgentTX Motivation: overhead accumulation and optimization

## Problem

This report turns the iterative runtime work into a paper-facing experiment. Trajectory-level isolation and causal recovery make every opaque tool call pay for tracing, script setup, snapshot traversal, and try namespace setup. The goal is to reduce that repeated cost without weakening the recovery boundary.

In [ ]:
import json
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
history = pd.read_csv(RESULTS / 'motivation_optimization_history.csv')
runtime = pd.read_csv(RESULTS / 'motivation_runtime_comparison.csv')
history

## What changed

The first stages remove redundant userspace work while preserving explicit dependency effects. The persistent worker addresses the largest repeated cost—creating and tearing down a try namespace for every call. Incremental snapshots then reduce upperdir traversal by cloning the previous state and replaying only changed paths; commit, rollback, reset, and resume retain a full-copy fallback.

In [ ]:
display(history[['iteration', 'optimization', 'metric', 'before', 'after', 'improvement_pct', 'correct']])
print('Largest endpoint reduction:')
display(history[history['iteration'] == 5][['optimization', 'before', 'after', 'improvement_pct']])

## Current comparison and tail behavior

The runtime comparison separates execution lower bounds from shared try and AgentTX modes. The deterministic workload is useful for attribution; the real-agent bundle below restores model decision-making and network latency for an end-to-end sanity check.

In [ ]:
display(runtime[['mode', 'per_step_mean_ms', 'wall_p50_s', 'wall_p95_s', 'host_polluted']])
with open(RESULTS / 'robustness.json', 'r', encoding='utf-8') as handle:
    robustness = json.load(handle)
with open(RESULTS / 'real_agent_robustness.json', 'r', encoding='utf-8') as handle:
    real_agent = json.load(handle)

runtime_tail = pd.DataFrame([row for row in robustness if row.get('suite') == 'p50_p95'])
display(runtime_tail[['mode', 'step_p50_ms', 'step_p95_ms', 'failure_rate']])
print({key: real_agent[key] for key in ['model', 'wall_p50_s', 'wall_p95_s', 'success_rate', 'host_leak_rate']})

## Correctness and motivation takeaway

AgentTX starts with a real systems tension: trajectory-level isolation and causal recovery add overhead to every tool boundary. The measurements decompose that cost and show why a persistent worker and incremental snapshot path are necessary. The correctness suite, crash injection, long-session reload, concurrent-agent isolation, and real-agent repeats ensure that the optimization story is not a throughput-only microbenchmark.